In [1]:
import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt
import pandas as pd
import os
import gc
import zipfile
from numba import njit
from tqdm import tqdm

# ==========================================
# 1. ネットワークと単体複体生成 (Numba JIT)
# ==========================================
@njit
def build_simplicial_complex(N, m):
    num_edges = 2 * m * N
    row = np.empty(num_edges, dtype=np.int32)
    col = np.empty(num_edges, dtype=np.int32)
    targets = np.empty(num_edges, dtype=np.int32)
    max_faces = m * m * N
    faces = np.empty((max_faces, 3), dtype=np.int32)

    idx, target_idx, face_idx = 0, 0, 0

    for i in range(m):
        for j in range(m):
            if i != j:
                row[idx] = i; col[idx] = j; idx += 1
                targets[target_idx] = i; target_idx += 1

    if m >= 3:
        faces[face_idx, 0] = 0; faces[face_idx, 1] = 1; faces[face_idx, 2] = 2; face_idx += 1

    chosen = np.empty(m, dtype=np.int32)

    for i in range(m, N):
        c_idx = 0
        while c_idx < m:
            r = np.random.randint(0, target_idx)
            node = targets[r]
            already = False
            for k in range(c_idx):
                if chosen[k] == node:
                    already = True; break
            if not already:
                chosen[c_idx] = node; c_idx += 1

        for j in range(m):
            for k in range(j + 1, m):
                faces[face_idx, 0] = i
                faces[face_idx, 1] = chosen[j]
                faces[face_idx, 2] = chosen[k]
                face_idx += 1

        for k in range(m):
            target = chosen[k]
            row[idx] = i; col[idx] = target; idx += 1
            row[idx] = target; col[idx] = i; idx += 1
            targets[target_idx] = i; target_idx += 1
            targets[target_idx] = target; target_idx += 1

    return row[:idx], col[:idx], faces[:face_idx]

@njit
def evaluate_faces(X, faces, active_faces, theta):
    ruptured = 0
    for f in range(faces.shape[0]):
        if active_faces[f]:
            i, j, k = faces[f, 0], faces[f, 1], faces[f, 2]
            frustration = max(X[i], X[j], X[k]) - min(X[i], X[j], X[k])
            if frustration > theta:
                active_faces[f] = False
                ruptured += 1
    return ruptured

# ==========================================
# 2. 1次元介入(エッジ追加)のシミュレーション関数
# ==========================================
def run_hypothesis_2_simulation(N, m, params):
    dt = params['dt']
    T1, T2, T3, T4 = params['T1'], params['T2'], params['T3'], params['T4']

    # トポロジー初期化
    row, col, faces = build_simplicial_complex(N, m)
    data = np.ones(len(row), dtype=np.float32)
    A_sparse = sp.csr_matrix((data, (row, col)), shape=(N, N))
    degrees = np.array(A_sparse.sum(axis=1)).flatten()
    degrees[degrees == 0] = 1.0
    hub_node = np.argmax(degrees)

    X = np.random.uniform(0, params['epsilon'], N).astype(np.float32)
    b = np.zeros(N, dtype=np.float32)

    num_faces = faces.shape[0]
    active_faces = np.ones(num_faces, dtype=bool)

    history_betti_1 = []

    # 状態変数
    ruptured_faces_count = 0
    added_edges_count = 0    # Phase 2で追加されるエッジ(1-単体)の数

    for step in tqdm(range(T4), desc=f"Simulating N={N}", leave=False):
        # フェーズ管理
        if step == T1:
            b[hub_node] = 0.9 # Phase 1a: フェイク投下
        elif step == T2:
            b[hub_node] = 0.0 # Phase 1b: フェイク除去（ヒステリシス）

        elif step == T3:
            # Phase 2: 1次元アルゴリズム介入（エッジのランダム追加）
            # プラットフォームが全エッジの10%相当のランダムなペアを推奨して接続
            M_new_edges = int(len(row) * 0.10)
            added_edges_count += M_new_edges

            # SDE力学系更新のための隣接行列アップデート (計算量抑制のためランダムに追加)
            new_row = np.random.randint(0, N, M_new_edges)
            new_col = np.random.randint(0, N, M_new_edges)
            new_data = np.ones(M_new_edges, dtype=np.float32)

            A_added = sp.csr_matrix((new_data, (new_row, new_col)), shape=(N, N))
            A_sparse = A_sparse + A_added
            degrees = np.array(A_sparse.sum(axis=1)).flatten()
            degrees[degrees == 0] = 1.0

        # SDEの更新
        P = A_sparse.dot(X) / degrees
        noise = np.random.normal(0, 1, N).astype(np.float32) * np.sqrt(dt)
        dX = (P + b - X) * dt + params['sigma'] * noise
        X = np.clip(X + dX, 0, 1)

        # 面の破裂を評価
        ruptured_faces_count += evaluate_faces(X, faces, active_faces, params['theta'])

        # Euler-Poincaré公式に基づく Betti 1 (穴) の計算
        # Δβ1 = 追加されたエッジ数(ΔE) + 破裂した面数(-ΔF)
        current_delta_betti_1 = added_edges_count + ruptured_faces_count

        # グラフスケールに合わせた正規化
        history_betti_1.append(current_delta_betti_1 / max(1, num_faces))

    return history_betti_1

# ==========================================
# 3. メイン実行スクリプト
# ==========================================
def main():
    # 1,000 から 10,000,000 までのマクロスケール
    N_list = [1000, 10000, 100000, 1000000, 10000000]

    params = {
        'dt': 0.1,
        'T1': 50,      # Phase 0: Burn-in 終了
        'T2': 150,     # Phase 1a: Fake News 終了
        'T3': 250,     # Phase 1b: Hysteresis 終了 -> 1D介入開始
        'T4': 350,     # Phase 2: 1D Intervention 終了
        'epsilon': 0.05,
        'sigma': 0.01,
        'theta': 0.3
    }

    output_dir = "Simplicial_TMFG_Hypothesis2"
    os.makedirs(output_dir, exist_ok=True)
    summary_data = []

    for N in N_list:
        try:
            print(f"\n[{N:,} Nodes] 仮説2 (1次元介入の敗北) シミュレーション開始...")
            betti_1_history = run_hypothesis_2_simulation(N, m=3, params=params)

            # CSV出力
            df = pd.DataFrame({
                'TimeStep': np.arange(params['T4']),
                'Delta_Betti_1_Ratio': betti_1_history
            })
            df.to_csv(os.path.join(output_dir, f'h2_result_N{N}.csv'), index=False)

            # グラフの出力
            plt.figure(figsize=(12, 6))
            plt.plot(df['TimeStep'], df['Delta_Betti_1_Ratio'], 'r-', linewidth=2.5, label='Topological Holes ($\Delta \\beta_1$)')

            # 背景色でフェーズを可視化
            plt.axvspan(0, params['T1'], color='gray', alpha=0.1, label='Phase 0: Burn-in')
            plt.axvspan(params['T1'], params['T2'], color='red', alpha=0.1, label='Phase 1a: Pathogenesis')
            plt.axvspan(params['T2'], params['T3'], color='blue', alpha=0.1, label='Phase 1b: Hysteresis')
            plt.axvspan(params['T3'], params['T4'], color='orange', alpha=0.2, label='Phase 2: 1D Intervention (Edge Injection)')

            plt.title(f'Failure of 1D Algorithmic Intervention (N={N:,})')
            plt.xlabel('Time Step')
            plt.ylabel('Normalized Topological Holes ($\Delta \\beta_1 / |F|$)')
            plt.legend(loc='upper left')
            plt.grid(True, alpha=0.3)
            plt.savefig(os.path.join(output_dir, f'h2_plot_N{N}.png'), dpi=300, bbox_inches='tight')
            plt.close()

            summary_data.append({
                'N': N,
                'Final_Betti_1_Ratio': betti_1_history[-1]
            })

            gc.collect()

        except MemoryError:
            print(f"[警告] N={N} でメモリ不足。ここまでのデータでZIP化へ移行します。")
            break

    # まとめグラフとZIP化
    if len(summary_data) > 0:
        df_sum = pd.DataFrame(summary_data)
        df_sum.to_csv(os.path.join(output_dir, 'h2_summary.csv'), index=False)

        plt.figure(figsize=(8, 6))
        plt.plot(df_sum['N'], df_sum['Final_Betti_1_Ratio'], 'k-o', linewidth=2, markersize=8)
        plt.xscale('log')
        plt.title('Asymptotic Topological Collapse Post-1D Intervention')
        plt.xlabel('Network Size N (Log Scale)')
        plt.ylabel('Final Topological Holes Proportion')
        plt.grid(True, alpha=0.3)
        plt.savefig(os.path.join(output_dir, 'h2_summary_plot.png'), dpi=300, bbox_inches='tight')
        plt.close()

        zip_filename = 'Simplicial_TMFG_Hypothesis2.zip'
        with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for root, _, files in os.walk(output_dir):
                for file in files:
                    zipf.write(os.path.join(root, file), arcname=file)

        print(f"\n🎉 仮説2の計算完了！結果は '{zip_filename}' に出力されました。")

if __name__ == "__main__":
    main()

<>:180: SyntaxWarning: invalid escape sequence '\D'
<>:190: SyntaxWarning: invalid escape sequence '\D'
<>:180: SyntaxWarning: invalid escape sequence '\D'
<>:190: SyntaxWarning: invalid escape sequence '\D'
/tmp/ipykernel_3356/1310560886.py:180: SyntaxWarning: invalid escape sequence '\D'
  plt.plot(df['TimeStep'], df['Delta_Betti_1_Ratio'], 'r-', linewidth=2.5, label='Topological Holes ($\Delta \\beta_1$)')
/tmp/ipykernel_3356/1310560886.py:190: SyntaxWarning: invalid escape sequence '\D'
  plt.ylabel('Normalized Topological Holes ($\Delta \\beta_1 / |F|$)')



[1,000 Nodes] 仮説2 (1次元介入の敗北) シミュレーション開始...



[10,000 Nodes] 仮説2 (1次元介入の敗北) シミュレーション開始...



[100,000 Nodes] 仮説2 (1次元介入の敗北) シミュレーション開始...



[1,000,000 Nodes] 仮説2 (1次元介入の敗北) シミュレーション開始...



[10,000,000 Nodes] 仮説2 (1次元介入の敗北) シミュレーション開始...



🎉 仮説2の計算完了！結果は 'Simplicial_TMFG_Hypothesis2.zip' に出力されました。


In [2]:
import numpy as np
import scipy.sparse as sp
import scipy.stats as stats
import matplotlib.pyplot as plt
import pandas as pd
import os
import gc
import zipfile
from numba import njit
from tqdm import tqdm

# ==========================================
# 1. ネットワークと単体複体生成 (Numba JIT)
# ==========================================
@njit
def build_simplicial_complex(N, m):
    num_edges = 2 * m * N
    row = np.empty(num_edges, dtype=np.int32)
    col = np.empty(num_edges, dtype=np.int32)
    targets = np.empty(num_edges, dtype=np.int32)
    max_faces = m * m * N
    faces = np.empty((max_faces, 3), dtype=np.int32)

    idx, target_idx, face_idx = 0, 0, 0

    for i in range(m):
        for j in range(m):
            if i != j:
                row[idx] = i; col[idx] = j; idx += 1
                targets[target_idx] = i; target_idx += 1

    if m >= 3:
        faces[face_idx, 0] = 0; faces[face_idx, 1] = 1; faces[face_idx, 2] = 2; face_idx += 1

    chosen = np.empty(m, dtype=np.int32)

    for i in range(m, N):
        c_idx = 0
        while c_idx < m:
            r = np.random.randint(0, target_idx)
            node = targets[r]
            already = False
            for k in range(c_idx):
                if chosen[k] == node:
                    already = True; break
            if not already:
                chosen[c_idx] = node; c_idx += 1

        for j in range(m):
            for k in range(j + 1, m):
                faces[face_idx, 0] = i
                faces[face_idx, 1] = chosen[j]
                faces[face_idx, 2] = chosen[k]
                face_idx += 1

        for k in range(m):
            target = chosen[k]
            row[idx] = i; col[idx] = target; idx += 1
            row[idx] = target; col[idx] = i; idx += 1
            targets[target_idx] = i; target_idx += 1
            targets[target_idx] = target; target_idx += 1

    return row[:idx], col[:idx], faces[:face_idx]

@njit
def evaluate_faces(X, faces, active_faces, theta):
    ruptured = 0
    for f in range(faces.shape[0]):
        if active_faces[f]:
            i, j, k = faces[f, 0], faces[f, 1], faces[f, 2]
            frustration = max(X[i], X[j], X[k]) - min(X[i], X[j], X[k])
            if frustration > theta:
                active_faces[f] = False
                ruptured += 1
    return ruptured

# ==========================================
# 2. 仮説2 シミュレーション (自己組織化のKS検定付き)
# ==========================================
def run_hypothesis2_with_validation(N, m, params):
    dt = params['dt']
    T1, T2, T3, T4 = params['T1'], params['T2'], params['T3'], params['T4']

    row, col, faces = build_simplicial_complex(N, m)
    data = np.ones(len(row), dtype=np.float32)
    A_sparse = sp.csr_matrix((data, (row, col)), shape=(N, N))
    degrees = np.array(A_sparse.sum(axis=1)).flatten()
    degrees[degrees == 0] = 1.0
    hub_node = np.argmax(degrees)

    del row, col, data
    gc.collect()

    X = np.random.uniform(0, params['epsilon'], N).astype(np.float32)
    b = np.zeros(N, dtype=np.float32)

    num_faces = faces.shape[0]
    active_faces = np.ones(num_faces, dtype=bool)

    history_betti_1 = []
    history_ks_stat = []
    X_snapshots = {}

    ruptured_faces_count = 0
    added_edges_count = 0
    X_prev = np.copy(X)

    for step in tqdm(range(T4), desc=f"Simulating N={N}", leave=False):
        if step == T1:
            b[hub_node] = 0.9
        elif step == T2:
            b[hub_node] = 0.0
        elif step == T3:
            M_new_edges = int(A_sparse.nnz * 0.05) # エッジを10%追加 (有向換算で5%)
            added_edges_count += M_new_edges

            new_row = np.random.randint(0, N, M_new_edges)
            new_col = np.random.randint(0, N, M_new_edges)
            new_data = np.ones(M_new_edges, dtype=np.float32)

            A_added = sp.csr_matrix((new_data, (new_row, new_col)), shape=(N, N))
            A_sparse = A_sparse + A_added
            degrees = np.array(A_sparse.sum(axis=1)).flatten()
            degrees[degrees == 0] = 1.0

        P = A_sparse.dot(X) / degrees
        noise = np.random.normal(0, 1, N).astype(np.float32) * np.sqrt(dt)
        dX = (P + b - X) * dt + params['sigma'] * noise
        X = np.clip(X + dX, 0, 1)

        ruptured_faces_count += evaluate_faces(X, faces, active_faces, params['theta'])

        # Euler-Poincaré: Δβ1 = ΔE - ΔF (ここでは ruptured_faces_count が破裂数なので加算)
        current_delta_betti_1 = added_edges_count + ruptured_faces_count
        history_betti_1.append(current_delta_betti_1 / max(1, num_faces))

        # Burn-in期間のKS検定と分布スナップショット
        if step <= T1:
            if step % 10 == 0:
                ks_stat, _ = stats.ks_2samp(X[::100], X_prev[::100])
                history_ks_stat.append((step, ks_stat))
                X_prev = np.copy(X)

            if step in [0, int(T1/2), T1]:
                X_snapshots[step] = np.copy(X)

    return history_betti_1, history_ks_stat, X_snapshots

# ==========================================
# 3. メイン実行スクリプト
# ==========================================
def main():
    N_list = [1000, 10000, 100000, 1000000]

    params = {
        'dt': 0.1,
        'T1': 100,     # Phase 0: Burn-in
        'T2': 200,     # Phase 1a: Fake News
        'T3': 300,     # Phase 1b: Hysteresis -> 1D Intervention
        'T4': 400,     # Phase 2: 1D Intervention Result
        'epsilon': 0.05,
        'sigma': 0.02,
        'theta': 0.25
    }

    output_dir = "Hypothesis2_Empirical_Validation"
    os.makedirs(output_dir, exist_ok=True)
    summary_data = []

    for N in N_list:
        try:
            print(f"\n[{N:,} Nodes] 仮説2 (1次元介入の敗北) ＋ 自己組織化検証 開始...")
            betti_1_hist, ks_hist, snapshots = run_hypothesis2_with_validation(N, 3, params)

            # --- グラフ1: KS統計量による自己組織化 ---
            steps_ks, ks_vals = zip(*ks_hist)
            plt.figure(figsize=(8, 5))
            plt.plot(steps_ks, ks_vals, 'g-o', linewidth=2)
            plt.title(f'Convergence to Empirical Stationary Distribution (N={N})')
            plt.xlabel('Burn-in Steps')
            plt.ylabel('KS Statistic (Distribution Drift)')
            plt.grid(True, alpha=0.3)
            plt.savefig(os.path.join(output_dir, f'H2_KS_Convergence_N{N}.png'), dpi=300)
            plt.close()

            # --- グラフ2: 状態分布の進化 ---
            plt.figure(figsize=(8, 5))
            for stp, x_data in snapshots.items():
                plt.hist(x_data, bins=50, alpha=0.5, density=True, label=f'Step {stp}')
            plt.title(f'Self-Organization of Conformity Distribution (N={N})')
            plt.xlabel('Conformity (X)')
            plt.ylabel('Density')
            plt.legend()
            plt.grid(True, alpha=0.3)
            plt.savefig(os.path.join(output_dir, f'H2_Distribution_Evolution_N{N}.png'), dpi=300)
            plt.close()

            # --- グラフ3: 仮説2 1次元介入によるBetti 1の爆発 ---
            df = pd.DataFrame({'TimeStep': np.arange(params['T4']), 'Delta_Betti_1_Ratio': betti_1_hist})
            df.to_csv(os.path.join(output_dir, f'h2_betti1_result_N{N}.csv'), index=False)

            plt.figure(figsize=(12, 6))
            plt.plot(df['TimeStep'], df['Delta_Betti_1_Ratio'], 'r-', linewidth=2.5, label='Topological Holes ($\Delta \\beta_1$)')
            plt.axvspan(0, params['T1'], color='gray', alpha=0.1, label='Phase 0: Burn-in (Self-Organization)')
            plt.axvspan(params['T1'], params['T2'], color='red', alpha=0.1, label='Phase 1a: Fake News Exposed')
            plt.axvspan(params['T2'], params['T3'], color='blue', alpha=0.1, label='Phase 1b: Fake Removed (Hysteresis)')
            plt.axvspan(params['T3'], params['T4'], color='orange', alpha=0.2, label='Phase 2: 1D Intervention (Edge Injection)')

            plt.title(f'Failure of 1D Algorithmic Intervention (N={N:,})')
            plt.xlabel('Time Step')
            plt.ylabel('Normalized Topological Holes ($\Delta \\beta_1 / |F|$)')
            plt.legend(loc='upper left')
            plt.grid(True, alpha=0.3)
            plt.savefig(os.path.join(output_dir, f'H2_Topological_Collapse_N{N}.png'), dpi=300)
            plt.close()

            summary_data.append({'N': N, 'Final_Betti_1_Ratio': betti_1_hist[-1]})
            gc.collect()

        except MemoryError:
            print(f"[警告] N={N} でメモリ不足。")
            break

    # まとめZIP化
    if summary_data:
        df_sum = pd.DataFrame(summary_data)
        df_sum.to_csv(os.path.join(output_dir, 'h2_summary_all_N.csv'), index=False)

        plt.figure(figsize=(8, 6))
        plt.plot(df_sum['N'], df_sum['Final_Betti_1_Ratio'], 'k-o', linewidth=2, markersize=8)
        plt.xscale('log')
        plt.title('Asymptotic Defect Explosion Post-1D Intervention')
        plt.xlabel('Network Size N (Log Scale)')
        plt.ylabel('Final Topological Holes Proportion')
        plt.grid(True, alpha=0.3)
        plt.savefig(os.path.join(output_dir, 'H2_summary_plot.png'), dpi=300)
        plt.close()

        zip_filename = 'Hypothesis2_Empirical_Validation.zip'
        with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for root, _, files in os.walk(output_dir):
                for file in files:
                    zipf.write(os.path.join(root, file), arcname=file)
        print(f"\n🎉 計算完了！結果は '{zip_filename}' に出力されました。")

if __name__ == "__main__":
    main()

<>:203: SyntaxWarning: invalid escape sequence '\D'
<>:211: SyntaxWarning: invalid escape sequence '\D'
<>:203: SyntaxWarning: invalid escape sequence '\D'
<>:211: SyntaxWarning: invalid escape sequence '\D'
/tmp/ipykernel_3356/1677476089.py:203: SyntaxWarning: invalid escape sequence '\D'
  plt.plot(df['TimeStep'], df['Delta_Betti_1_Ratio'], 'r-', linewidth=2.5, label='Topological Holes ($\Delta \\beta_1$)')
/tmp/ipykernel_3356/1677476089.py:211: SyntaxWarning: invalid escape sequence '\D'
  plt.ylabel('Normalized Topological Holes ($\Delta \\beta_1 / |F|$)')



[1,000 Nodes] 仮説2 (1次元介入の敗北) ＋ 自己組織化検証 開始...



[10,000 Nodes] 仮説2 (1次元介入の敗北) ＋ 自己組織化検証 開始...



[100,000 Nodes] 仮説2 (1次元介入の敗北) ＋ 自己組織化検証 開始...



[1,000,000 Nodes] 仮説2 (1次元介入の敗北) ＋ 自己組織化検証 開始...



🎉 計算完了！結果は 'Hypothesis2_Empirical_Validation.zip' に出力されました。
